## Observing the CMB

In [ ]:
import maria
from maria.band import get_band

f090 = get_band("act/pa5/f090")
f150 = get_band("act/pa5/f150")

f090.NET_RJ = 20e-6
f150.NET_RJ = 20e-6

f090.knee = 1e1
f150.knee = 1e1

array = {"field_of_view": 0.25,
         "primary_size": 10,
         "n": 80,
         "packing": "sunflower",
         "shape": "circle",
         "polarized": True,
         "bands": [f090, f150]}

instrument = maria.get_instrument(array=array)

print(instrument)
instrument.plot()

In [ ]:
from maria import Plan

plan1 = Plan.generate(duration=1200, 
                     sample_rate=25, 
                     start_time="2026-08-05T06:00:00",
                     scan_type="back-and-forth",
                     scan_parameters={"az_center": 90,
                                      "el_center": 45,
                                      "az_throw": 4,
                                     },
                     site="cerro_toco")

plan1.plot(frames=["az/el", "ra/dec", "glon/glat"])

plan2 = Plan.generate(duration=1200, 
                     sample_rate=25, 
                     start_time="2026-08-05T12:18:00",
                     scan_type="back-and-forth",
                     scan_parameters={"az_center": -90,
                                      "el_center": 45,
                                      "az_throw": 4,
                                     },
                     site="cerro_toco")

plan2.plot(frames=["az/el", "ra/dec", "glon/glat"])

In [ ]:
sim = maria.Simulation(
    instrument=instrument,
    plans=[plan1, plan2],
    site="cerro_toco",
    cmb="generate",
    cmb_kwargs={"nside": 1024},
)

print(sim)

In [ ]:
tods = sim.run()
tods[0].plot()

Binning the data gives us a 

In [ ]:
from maria.mappers import *

plot_kwargs = {"slices": dict(stokes=["I", "Q", "U"], nu=[[0], [1]]), "contrast": 1e-2}

bin_mapper = BinMapper(tods=tods,
                       units="uK_CMB",
                       resolution=2 / 60,
                       frame="ra/dec",
                       tod_preprocessing={
                        "remove_polynomial": {"time": 3, "elevation": 3},
                       },
                      )

bin_mapper.run()
bin_mapper.map.plot(**plot_kwargs)

In [ ]:
from maria.mapping.ml_mapper import *

ml_mapper = MaximumLikelihoodMapper(tods=tods,
                                    units="uK_CMB",
                                    resolution=1 / 60,
                                    frame="ra/dec",
                                    tod_preprocessing={
                                    "remove_polynomial": {"time": 3, "elevation": 3},
                                   },
                                   )

In [ ]:
print(ml_mapper.map)
ml_mapper.map.plot(**plot_kwargs)

In [ ]:
ml_mapper.fit(epochs=1, 
              max_steps_per_epoch=50, 
              plot=True, 
              plot_kwargs=plot_kwargs)